<a href="https://colab.research.google.com/github/saranyav02/Computational_Pharmacology/blob/main/pinn_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
from collections import OrderedDict

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Device is {device}")

Device is cpu


Consider the two compartment model introduced in Module I. Recall that this model divides the body into a central compartment and a peripheral compartment. Drug transfer between compartments is modelled as
\begin{align*}
    \frac{dC_p}{dt}&=k_{cp}C_c-k_{pc}C_p\\
    \frac{dC_c}{dt}&=k_a+k_{pc}C_p-k_{cp}C_c-{k_e}C_c\\
    \frac{dA}{dt}&=-k_aA
\end{align*}


In [ ]:
#define model parameters
k_cp=1
k_pc=1
k_a=1
k_e=0.5


def dudt(t,u):
#------------------------------------------------------------------------------
#YOUR CODE HERE
#-----------------------------------------------------------------------------

#equation parameters
max_t = 10
step_size = 1.0
noise = 0.0
t = np.arange(0, max_t, step_size)

#define initial conditions and solve IVP
#------------------------------------------------------------------------------
#YOUR CODE HERE
#-----------------------------------------------------------------------------
#ideal data
#------------------------------------------------------------------------------
#YOUR CODE HERE
#-----------------------------------------------------------------------------
#add noise
#------------------------------------------------------------------------------
#YOUR CODE HERE
#-----------------------------------------------------------------------------

#training and test data
#------------------------------------------------------------------------------
#YOUR CODE HERE
#-----------------------------------------------------------------------------

#define collocation points
t_colloc = np.expand_dims(np.arange(0, max_t, 0.001),axis=1)



#plot synthetic data

tm, tM = t_train.min(), t_train.max()
ym, yM = y_train.min(axis=0), y_train.max(axis=0)
y_train=y_train.transpose()
print(y_train.shape)
plt.grid()
plt.title("Trajectories")
plt.plot(t_train, y_train[:,0], 'o',label='value',
            color='k')
plt.plot(t_train, y_train[:,1], 'o',label='value',
            color='b')
plt.plot(t_train, y_train[:,2], 'o',label='value',
            color='r')
plt.legend()
plt.show()

print('number of data points:',y_train.shape[0]) #number of data points
print('number of collocation points:',t_colloc.shape[0]) #collocation points




Question 2: Construct the feed forward neural network

In [ ]:
class DNN(torch.nn.Module):
    def __init__(self, layers, min_val, max_val):
        super(DNN, self).__init__()

        # parameters
        self.depth = len(layers) - 1

        self.min_val = torch.tensor([min_val], requires_grad=True).float().to(device) #changed requires grad from true
        self.max_val = torch.tensor([max_val], requires_grad=True).float().to(device)

        # set up layer order dict
        self.activation = torch.nn.Tanh

        layer_list = list()
        #------------------------------------------------------------------------------
        #YOUR CODE HERE
        #-----------------------------------------------------------------------------
        layerDict = OrderedDict(layer_list)

        # deploy layers
        self.layers = torch.nn.Sequential(layerDict)


    def forward(self, x):
        res = (x - self.min_val) / (self.max_val - self.min_val)
        #------------------------------------------------------------------------------
        #YOUR CODE HERE
        #-----------------------------------------------------------------------------

        return out

max_lr = -1
min_lr = -3

time_delta = 30000
warm_ups = 1500
rate = (min_lr - max_lr) / time_delta

def lr_schedule(epoch):
    """Linear annealing LR schedule."""
    if epoch < warm_ups:
        return 10**(max_lr)
    elif epoch < time_delta:
      return 10**(rate * (epoch - warm_ups) + max_lr)
    else:
        return 10**min_lr

Question 3: Suppose we want to recover $k_{cp}$. Construct the physics informed neural network

In [ ]:
#parameters to be learned
#p_= YOUR CODE HERE

#PINN Class
class PhysicsInformedNN():
    def __init__(self, t, t_colloc, u_data, p_, layers, lb, ub):

        # data
        self.x = torch.tensor(X, requires_grad=True).float().to(device)

        self.t = torch.tensor(t, requires_grad=True).float().to(device)
        self.t_colloc = torch.tensor(t_colloc, requires_grad=True).float().to(device)
        self.u_data = torch.tensor(u_data, requires_grad=False).float().to(device).reshape(-1, 1)

        #define trainable parameters
        k_cp = p_
        self.k_cp = torch.tensor([k_cp], requires_grad=True).float().to(device)

        self.k_cp = torch.nn.Parameter(self.k_cp)

        # deep neural networks
        print(f"layers: {layers}")
        self.dnn = DNN(layers,tm,tM).to(device)
        self.dnn.register_parameter('k_cp', self.k_cp)


         # optimizers: using the same settings
        self.optimizer = torch.optim.LBFGS(
            self.dnn.parameters(),
            lr=1.0,
            max_iter=50000,
            max_eval=50000,
            history_size=50,
            tolerance_grad=0,
            tolerance_change=0,
            line_search_fn="strong_wolfe"
        )

        self.optimizer_Adam = torch.optim.Adam(self.dnn.parameters())
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer_Adam, lr_lambda=lr_schedule)
        self.iter = 0

    def net_u(self, t):
        u = self.dnn(t)
        return u

    #define the PINN loss
    def net_f(self, t, t_colloc, u_data):
        """ The pytorch autograd version of calculating residual """
        u = self.net_u(t).float().to(device).reshape(-1, 1)
        u_colloc = self.net_u(t_colloc)
        loss_mse = torch.mean(torch.square(u - u_data))

        _Cp = torch.unsqueeze(u_colloc[:, 0],axis=1)
        _Cc = torch.unsqueeze(u_colloc[:, 1],axis=1)
        _A = torch.unsqueeze(u_colloc[:, 2],axis=1)

        #define the network derivatives
        #------------------------------------------------------------------------------
        #YOUR CODE HERE
        #-----------------------------------------------------------------------------

        # Right-hand side
        #Cp_dot=YOUR CODE HERE
        #Cc_dot=YOUR CODE HERE
        #A_dot=YOUR CODE HERE
        x_dot=torch.cat((Cp_dot,Cc_dot,A_dot),axis=1)

        loss_pinn = torch.mean(torch.square(x_t - x_dot))

        return loss_pinn
    def net_ic(self):
        t_ic = torch.tensor([[0.0]], requires_grad=True).float().to(device)
        x_ic=self.net_u(t_ic)

        ic_preds=[x_ic]
        ic_preds=[x.cpu().detach().numpy() for x in ic_preds]
        #ic_loss = #YOUR CODE HERE
        return ic_loss

    def loss_func(self):
        self.optimizer.zero_grad()


        #pinn_loss = YOUR CODE HERE
        #ic_loss=YOUR CODE HERE
        u_pred = self.net_u(self.t).float().to(device).reshape(-1, 1)
        #data_loss = YOUR CODE HERE

        #loss = YOUR CODE HERE

        loss.backward()

        self.iter += 1
        if self.iter % 500 == 0:
            print(
                'Loss: %e, data_loss: %e, pinn_loss: %e, ic_loss: %e, k_cp: %.3f' %
                (
                    loss.item(),
                    data_loss.item(),
                    pinn_loss.item(),
                    ic_loss.item(),
                    self.k_cp.item(),

                )
            )
            self.losses.append(loss.item())
            self.k_cp_vals.append(self.k_cp.item())

        return loss


    def train(self, adam_epochs, bfgs_epochs, polish_adam_epochs):
        self.dnn.train()
        self.losses=[]
        self.k_cp_vals=[]
        for epoch in range(adam_epochs):
            u_pred = self.net_u(self.t).float().to(device).reshape(-1, 1)
            loss_pinn = self.net_f(self.t, self.t_colloc, self.u_data)
            loss_mse = torch.mean(torch.square(self.u_data - u_pred))
            loss_ic=self.net_ic()
            loss = loss_mse + loss_pinn+loss_ic

            # Backward and optimize
            self.optimizer_Adam.zero_grad()
            loss.backward()

            self.optimizer_Adam.step()
            self.scheduler.step()

            if epoch % 500 == 0:
                print(
                    'It: %d, Loss: %.3e , loss_mse: %.3e,  k_cp: %.3f' %
                    (
                        epoch,
                        loss.item(),
                        loss_mse.item(),
                        self.k_cp.item(),
                    )
                )
            self.losses.append(loss.item())
            self.k_cp_vals.append(self.k_cp.item())
        self.optimizer.step(self.loss_func)

        plt.plot(np.array(self.losses))
        plt.yscale('log')
        plt.xlabel('Epochs')
        plt.ylabel('Log Loss')
        plt.show()


        plt.plot(np.array(self.k_cp_vals),label='Predicted k_cp value')
        plt.plot( k_cp*np.ones_like(np.array(self.k_cp_vals)),'--', label='PINN target k_cp value', color='r')
        plt.xlabel('Epochs')
        plt.ylabel('k_cp Value')
        ax = plt.gca()
        ax.set_ylim([0, 1])
        plt.legend()
        plt.show()
    def predict(self, t):
        t = torch.tensor(t, requires_grad=True).float().to(device)

        self.dnn.eval()
        u = self.net_u(t)
        u = u.detach().cpu().numpy()
        return u

In [ ]:

#layers = YOUR CODE HERE

"""## Training on Non-noisy Data"""
adam_epochs = 2000
polish_adam_epochs = 1000
bfgs_epochs = 100
# training
#model = YOUR CODE HERE
model.train(adam_epochs, bfgs_epochs, polish_adam_epochs)
u_pred= model.predict(model.t)



In [ ]:
#visualize data predictions
#------------------------------------------------------------
#YOUR CODE HERE
#------------------------------------------------------------